# 02 · Holds, coordinates and climbs

The board only knows **LED positions** (0, 1, 2, ...). The Climbdex database knows
where each LED's hold sits on the wall (x/y in inches) and which holds make up each climb.

Needs the database: run `bin/sync_db.sh kilter` (Windows: `bin/win/sync_db.sh kilter`) once.

In [ ]:
# Make the repository root importable when the notebook runs from examples/
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from boardlights import Board, Wall, scan

In [ ]:
# --- Configure your wall here ------------------------------------------------
# IICW default: Kilter Homewall (layout 8), 7x10 Mainline LED kit (size 18, 165 LEDs).
# Not sure? Run `python -m boardlights configs` to list all layout/size ids.
LAYOUT_ID = 8
SIZE_ID = 18
BOARD_ADDRESS = None   # e.g. "FC:0C:87:2B:53:25"; None = connect to the closest board

In [ ]:
wall = Wall(LAYOUT_ID, SIZE_ID)
min_x, max_x, min_y, max_y = wall.bounds()
print(f"{len(wall.positions)} LEDs, x from {min_x} to {max_x}, y from {min_y} to {max_y} (inches)")
wall.unique_holds()[:5]

## A text map of the wall
Each number is an LED position, laid out by its real x/y (top of the wall at the top).

In [ ]:
holds = wall.unique_holds()
xs = sorted({h.x for h in holds})
ys = sorted({h.y for h in holds}, reverse=True)
grid = {(h.x, h.y): h.position for h in holds}
for y in ys:
    print(" ".join(f"{grid[(x, y)]:>3}" if (x, y) in grid else "  ." for x in xs))

In [ ]:
board = await Board.connect(address=BOARD_ADDRESS)

## Light holds by where they are
For example: the bottom row, the left column, or everything above head height.

In [ ]:
bottom = {h.position: "00FF00" for h in holds if h.y == min_y}
left   = {h.position: "0000FF" for h in holds if h.x == min_x}
await board.show({**bottom, **left})

In [ ]:
# Everything higher than 100 inches (~2.5 m) in orange
await board.show({h.position: "FFA500" for h in holds if h.y > 100})

In [ ]:
# The hold closest to the middle of the wall
middle = wall.nearest((min_x + max_x) / 2, (min_y + max_y) / 2)
print(middle)
await board.show({middle.position: "FF00FF"})

## Light a real climb
Climbs are stored as a *frames* string like `p4119r45p4139r45...`:
`p<placement>r<role>`, where the role is start (green), hand (cyan), finish (magenta) or foot (orange).

In [ ]:
climbs = wall.find_climbs("", limit=10)    # put part of a name between the quotes
for c in climbs:
    print(c.name)

In [ ]:
climb = climbs[0]
print("Lighting", climb.name)
await board.show(wall.climb_leds(climb))

## Exercise: set your own problem in code
Pick holds by coordinates, give them roles, and light them. Bonus: make a function that
generates a random "ladder" climb from bottom to top.

In [ ]:
START, HAND, FINISH, FOOT = "00FF00", "00FFFF", "FF00FF", "FFA500"
my_climb = {
    wall.nearest(0, 40).position: START,
    wall.nearest(-16, 70).position: HAND,
    wall.nearest(16, 100).position: HAND,
    wall.nearest(0, 140).position: FINISH,
}
await board.show(my_climb)

In [ ]:
await board.clear()
await board.disconnect()